# Intermediate 01 - 변환 함수 (Transformations)

이 노트북에서는:
- Kamailio의 변환 함수(transformations)를 배웁니다
- URI 분해, 문자열 조작, 정규식 등 실무에 필수적인 변환을 실습합니다

---

## 변환 함수란?

변환 함수는 변수 뒤에 `{변환명}`을 붙여 값을 가공합니다:

```
$(변수{변환명})
```

예: `$(ru{uri.user})` → `$ru`에서 사용자 부분만 추출

## 1. URI 변환

SIP URI에서 각 부분을 추출하는 것은 가장 많이 쓰는 변환입니다.

In [1]:
%%sip INVITE
From: <sip:1001@10.60.91.199:5060>;tag=abc
To: <sip:1002@10.60.80.218:5060>

Mock INVITE message created:


INVITE sip:1002@10.60.80.218:5060 SIP/2.0
From: <sip:1001@10.60.91.199:5060>;tag=abc
To: <sip:1002@10.60.80.218:5060>
Call-ID: e5562f35@mock
CSeq: 1 INVITE


Variables initialized:


  $Ri = "127.0.0.1"


  $Rp = 5060


...

  $ru = "sip:1002@10.60.80.218:5060"


  $si = "127.0.0.1"


  $sp = 5060


  $tu = "sip:1002@10.60.80.218:5060"


  $ua = ""


In [2]:
$var(uri) = $ru;
xlog("Full URI: $var(uri)");
xlog("User part: $(var(uri){uri.user})");
xlog("Host part: $(var(uri){uri.host})");
xlog("Port part: $(var(uri){uri.port})");

$var(uri) = "sip:1002@10.60.80.218:5060"  (string)


[LOG] Full URI: $var(uri)


[LOG] User part: $(var(uri){uri.user})


[LOG] Host part: $(var(uri){uri.host})


[LOG] Port part: $(var(uri){uri.port})


Variables changed:


  $var(uri): <null> → "sip:1002@10.60.80.218:5060"


## 2. From URI에서 발신자 추출

실무에서는 발신자 ID를 추출해서 로깅이나 라우팅 결정에 사용합니다.

In [3]:
$var(caller) = $(fu{uri.user});
$var(callee) = $(ru{uri.user});
xlog("Call from $var(caller) to $var(callee)");

$var(caller) = "1001"  (string)


$var(callee) = "1002"  (string)


[LOG] Call from $var(caller) to $var(callee)


Variables changed:


  $var(caller): <null> → "1001"


  $var(callee): <null> → "1002"


## 3. 문자열 변환

문자열 길이, 대소문자 변환 등의 변환도 자주 사용됩니다.

In [4]:
$var(name) = "Alice";

# 길이
xlog("Length: $(var(name){s.len})");

# 대문자
xlog("Upper: $(var(name){s.upper})");

# 소문자
xlog("Lower: $(var(name){s.lower})");

$var(name) = "Alice"  (string)


[LOG] Length: $(var(name){s.len})


[LOG] Upper: $(var(name){s.upper})


[LOG] Lower: $(var(name){s.lower})


Variables changed:


  $var(name): <null> → "Alice"


## 4. 문자열 → 정수 변환

문자열에서 포트 번호를 추출한 후 정수로 변환하는 패턴이 자주 쓰입니다.

In [5]:
$var(port_str) = "5060";
$var(port) = $(var(port_str){s.int});

xlog("Port as string: $var(port_str)");
xlog("Port as integer: $var(port)");

if ($var(port) == 5060) {
    xlog("Standard SIP port");
}

$var(port_str) = "5060"  (string)


$var(port) = "<undefined: $(var(port_str))>"  (string)


[LOG] Port as string: $var(port_str)


[LOG] Port as integer: $var(port)


✗ if ($var(port) == 5060) → FALSE


  🔀 if($var(port) == 5060): FALSE


Variables changed:


  $var(port_str): <null> → "5060"


  $var(port): <null> → "<undefined: $(var(port_str))>"


## 5. Nameaddr 변환

From/To 헤더는 `"Display Name" <sip:user@host>;tag=xxx` 형식입니다.
각 부분을 추출하는 변환을 배워봅니다.

In [6]:
$var(from_hdr) = '"Alice" <sip:1001@example.com>;tag=abc123';

# URI 부분만 추출
xlog("URI: $(var(from_hdr){nameaddr.uri})");

# Display name 추출
xlog("Name: $(var(from_hdr){nameaddr.name})");

# 파라미터 추출
xlog("Params: $(var(from_hdr){nameaddr.params})");

$var(from_hdr) = "'"Alice" <sip:1001@example.com>;tag=abc123'"  (string)


[LOG] URI: $(var(from_hdr){nameaddr.uri})


[LOG] Name: $(var(from_hdr){nameaddr.name})


[LOG] Params: $(var(from_hdr){nameaddr.params})


Variables changed:


  $var(from_hdr): <null> → "'"Alice" <sip:1001@example.com>;tag=abc123'"


## 6. IP 주소 변환

IP 주소가 유효한지 확인하는 변환입니다.

In [7]:
$var(ip1) = "10.60.91.199";
$var(ip2) = "not-an-ip";

# 유효한 IP인지 확인 (1=valid, 0=invalid)
xlog("IP1 valid: $(var(ip1){ip.isip})");
xlog("IP2 valid: $(var(ip2){ip.isip})");

$var(ip1) = "10.60.91.199"  (string)


$var(ip2) = "not-an-ip"  (string)


[LOG] IP1 valid: $(var(ip1){ip.isip})


[LOG] IP2 valid: $(var(ip2){ip.isip})


Variables changed:


  $var(ip1): <null> → "10.60.91.199"


  $var(ip2): <null> → "not-an-ip"


## 7. 변환 체이닝

여러 변환을 연속으로 적용할 수 있습니다.
예: URI에서 user를 추출한 다음 대문자로 변환

In [8]:
$var(test_uri) = "sip:alice@example.com";

# user 추출 후 대문자
xlog("Upper user: $(var(test_uri){uri.user})");

$var(test_uri) = "sip:alice@example.com"  (string)


[LOG] Upper user: $(var(test_uri){uri.user})


Variables changed:


  $var(test_uri): <null> → "sip:alice@example.com"


## 8. Base64 인코딩/디코딩

SIP 메시지에 바이너리 데이터를 넣을 때 유용합니다.

In [9]:
$var(original) = "Hello Kamailio";
$var(encoded) = $(var(original){s.encode.base64});
xlog("Encoded: $var(encoded)");

$var(decoded) = $(var(encoded){s.decode.base64});
xlog("Decoded: $var(decoded)");

$var(original) = "Hello Kamailio"  (string)


$var(encoded) = "<undefined: $(var(original))>"  (string)


[LOG] Encoded: $var(encoded)


$var(decoded) = "<undefined: $(var(encoded))>"  (string)


[LOG] Decoded: $var(decoded)


Variables changed:


  $var(original): <null> → "Hello Kamailio"


  $var(encoded): <null> → "<undefined: $(var(original))>"


  $var(decoded): <null> → "<undefined: $(var(encoded))>"


---

### 변환 함수 요약

| 카테고리 | 변환 | 설명 |
|----------|------|------|
| URI | `{uri.user}` | 사용자 부분 |
| URI | `{uri.host}` | 호스트 부분 |
| URI | `{uri.port}` | 포트 번호 |
| 문자열 | `{s.len}` | 길이 |
| 문자열 | `{s.upper}` / `{s.lower}` | 대소문자 |
| 문자열 | `{s.int}` | 정수 변환 |
| Nameaddr | `{nameaddr.uri}` | URI 추출 |
| Nameaddr | `{nameaddr.name}` | 표시명 추출 |
| IP | `{ip.isip}` | IP 유효성 |
| 인코딩 | `{s.encode.base64}` | Base64 인코드 |
| 인코딩 | `{s.decode.base64}` | Base64 디코드 |

다음 노트북: **02-dispatcher-and-routing.ipynb** →